# RentFleet J15-B — audit reproductible de la référence HGB

Ce notebook public **ne réouvre pas** le test final et ne réentraîne pas le modèle. Il vérifie le paquet scientifique publié, son protocole chronologique et sa décision. Le benchmark Munich prouve la méthode, jamais une accuracy locale RentFleet. L'exécution Colab J5 canonique et sa provenance complète restent archivées séparément.

In [ ]:
from pathlib import Path
import hashlib, json, platform, subprocess
REPO = Path('/content/pfe')
if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        '--branch', 'science/fleet-reallocation-qualification',
        'https://github.com/getibplay-cmyk/pfe.git', str(REPO),
    ], check=True)
EVIDENCE = REPO / 'docs/evidence/intelligence/demand-forecast'
assert EVIDENCE.is_dir(), EVIDENCE
print(platform.python_version())

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
checked = []
for line in (EVIDENCE / 'SHA256SUMS').read_text(encoding='utf-8').splitlines():
    expected, name = line.split('  ', 1)
    assert sha256(EVIDENCE / name) == expected, name
    checked.append(name)
print('Preuves publiées vérifiées:', len(checked))

In [ ]:
# Le test final reste intact : audit de la décision existante, aucune fonction fit().
DO_NOT_REOPEN_FINAL_TEST = True
decision = json.loads((EVIDENCE / 'qualification-manifest.json').read_text(encoding='utf-8'))
assert decision['final_test']['confirmation_gate_passed'] is True
assert decision['selection']['final_test_used_for_retuning'] is False
assert decision['model']['identifier'] == 'hgb_poisson::regularized'
assert abs(decision['final_test']['wape'] - 0.15234192004813368) < 1e-15
assert abs(decision['final_test']['mase'] - 0.8295561180756534) < 1e-15
assert decision['local_status'] == 'not_validated_without_sufficient_real_rentfleet_history'
assert decision['integration']['automatic_operational_action'] is False
decision['decision']

## Interprétation obligatoire

Le HGB reste une **référence consultative D+1 à D+7**. Il ne modifie automatiquement aucune réservation, contrat, tarification, facture, véhicule ou réallocation. Une validation humaine est obligatoire. L'absence d'un historique réel RentFleet suffisant maintient le statut local à **non validé**.